# RQ5 earliest actionable detection

**Research question:** At what earliest point in a course can a causal learning analytics model deliver reliable and actionable risk detection without sacrificing predictive performance?

This notebook is designed for Kaggle execution and saves all result tables as CSV files and figures as PDF files under `/kaggle/working/results/rq5_earliest_actionable_detection`.

In [1]:
import os, sys, json, math, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path('/kaggle/input/datasets/kimdaligermany/seoyeon-thesis-src')
DATASET_PATH = '/kaggle/input/datasets/anlgrbz/student-demographics-online-education-dataoulad'

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.config import ExperimentConfig
from src.data_utils import (load_generic_education_dataset,
                             build_weekly_timelines, derive_targets,
                             cumulative_snapshot, make_sequence_tensor)
from src.feature_engineering import numeric_feature_columns, make_preprocessor
from src.models import (fit_dual_tabular_model, predict_dual_tabular_model,
                        LSTMClassifier, TransformerClassifier,
                        MultiTaskCausalNet, train_torch_model,
                        predict_torch_multitask)
from src.evaluation import (classification_summary, summarise_dual_task,
                             topk_alert_precision, expected_calibration_error)
from src.causal_utils import (correlation_dag, direct_indirect_effects,
                               bootstrap_edge_stability)
from src.counterfactual_utils import (generate_simple_counterfactuals,
                                      evaluate_counterfactuals,
                                      segment_joint_risk)
from src.plotting_utils import lineplot, scatterplot, heatmap, barplot

CFG = ExperimentConfig()
DATASET_PATH = '/kaggle/input/datasets/anlgrbz/student-demographics-online-education-dataoulad'
CFG.dataset_name = "rq5_earliest_actionable_detection"
OUT = CFG.output_dir
print("Output directory:", OUT)

Output directory: /kaggle/working/results/rq5_earliest_actionable_detection


In [2]:
# Load raw logs, normalize them into a weekly timeline, and derive labels.
raw_df    = load_generic_education_dataset(DATASET_PATH,
                                           dataset_name=CFG.dataset_name)
weekly_df = build_weekly_timelines(raw_df)
weekly_df = derive_targets(weekly_df)
weekly_df.head()

,student_id,week,code_module,code_presentation,sum_click,n_activities,n_submissions,mean_score,late_rate,gender,age_band,imd_band,highest_education,num_of_prev_attempts,studied_credits,dropout,failure,final_result
0,6516,0,AAA,2014J,485,89,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
1,6516,1,AAA,2014J,42,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
2,6516,2,AAA,2014J,79,20,1.0,0.6,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
3,6516,3,AAA,2014J,193,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
4,6516,4,AAA,2014J,69,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass


## RQ5 workflow

This notebook identifies the earliest operationally useful course week for reliable and actionable risk detection.

In [3]:
week_rows = []
max_week  = weekly_df["week"].max()

for wk in CFG.early_weeks:
    snap         = cumulative_snapshot(weekly_df, up_to_week=wk)
    feature_cols = numeric_feature_columns(snap)
    X            = snap[feature_cols].fillna(0)
    y_drop       = snap["dropout"].astype(int).values
    y_fail       = snap["failure"].astype(int).values

    X_tr, X_te, yd_tr, yd_te, yf_tr, yf_te = train_test_split(
        X, y_drop, y_fail,
        test_size=0.25, random_state=CFG.random_state, stratify=y_drop)

    prep  = make_preprocessor()
    X_trp = prep.fit_transform(X_tr)
    X_tep = prep.transform(X_te)

    model         = fit_dual_tabular_model(
        X_trp, yd_tr, yf_tr,
        name="xgboost", random_state=CFG.random_state)
    p_drop, p_fail = predict_dual_tabular_model(model, X_tep)

    metrics_drop = classification_summary(yd_te, p_drop)
    ece_val      = expected_calibration_error(yd_te, p_drop)
    lead_time    = max(0, int(max_week) - wk)

    week_rows.append({
        "week":           wk,
        "AUROC":          metrics_drop["AUROC"],
        "Precision":      metrics_drop["Precision"],
        "Recall":         metrics_drop["Recall"],
        "ECE":            ece_val,
        "lead_time_weeks": lead_time,
    })

week_df = pd.DataFrame(week_rows)
week_df.to_csv(
    OUT / "table_5_1_weekly_intervention_readiness_metrics.csv", index=False)
week_df

,week,AUROC,Precision,Recall,ECE,lead_time_weeks
0,2,0.7431,0.6792,0.2593,0.0123,36
1,4,0.7791,0.6915,0.3998,0.0145,34
2,6,0.7842,0.6635,0.4336,0.0158,32
3,8,0.8113,0.7128,0.4948,0.0142,30
4,10,0.8067,0.6982,0.4878,0.0146,28
5,12,0.8190,0.7081,0.4878,0.0172,26


In [4]:
# Figure 5.1 Early-warning utility over course time
utility_df = week_df.copy()
utility_df["operational_utility"] = (
    0.5 * utility_df["AUROC"]
    + 0.3 * utility_df["Precision"]
    + 0.2 * utility_df["lead_time_weeks"]
          / max(1, utility_df["lead_time_weeks"].max())
)

plt.figure(figsize=(8, 5))
plt.plot(utility_df["week"], utility_df["operational_utility"],
         marker="o", label="Operational utility",
         color="#2563EB", linewidth=2.2)
plt.plot(utility_df["week"], utility_df["AUROC"],
         marker="s", label="AUROC",
         color="#DC2626", linewidth=2.2)
plt.title("Figure 5.1 Early-warning utility over course time",
          fontweight="bold")
plt.xlabel("Week")
plt.ylabel("Score")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "figure_5_1_early_warning_utility_over_course_time.pdf",
            bbox_inches="tight", facecolor="white")
plt.close()

utility_df

,week,AUROC,Precision,Recall,ECE,lead_time_weeks,operational_utility
0,2,0.7431,0.6792,0.2593,0.0123,36,0.775310
1,4,0.7791,0.6915,0.3998,0.0145,34,0.785889
2,6,0.7842,0.6635,0.4336,0.0158,32,0.768928
3,8,0.8113,0.7128,0.4948,0.0142,30,0.786157
4,10,0.8067,0.6982,0.4878,0.0146,28,0.768366
5,12,0.8190,0.7081,0.4878,0.0172,26,0.766374


In [5]:
work_rows    = []
surface_rows = []

for wk in CFG.early_weeks:
    snap_w       = cumulative_snapshot(weekly_df, up_to_week=wk)
    feature_cols = numeric_feature_columns(snap_w)
    X            = snap_w[feature_cols].fillna(0)
    y            = snap_w["dropout"].astype(int).values
    y_fail       = snap_w["failure"].astype(int).values

    X_tr, X_te, y_tr, y_te, yf_tr, yf_te = train_test_split(
        X, y, y_fail,
        test_size=0.25, random_state=CFG.random_state, stratify=y)

    prep  = make_preprocessor()
    X_trp = prep.fit_transform(X_tr)
    X_tep = prep.transform(X_te)

    model        = fit_dual_tabular_model(
        X_trp, y_tr, yf_tr,
        name="xgboost", random_state=CFG.random_state)
    probs, _     = predict_dual_tabular_model(model, X_tep)

    for thr in np.arange(0.3, 0.8, 0.1):
        thr           = round(float(thr), 1)
        alerts        = (probs >= thr).astype(int)
        alerts_sent   = int(alerts.sum())
        true_at_risk  = int(((alerts == 1) & (y_te == 1)).sum())
        false_alerts  = int(((alerts == 1) & (y_te == 0)).sum())
        advisor_hours = round(alerts_sent * 0.17, 2)

        surface_rows.append({
            "week":                 wk,
            "threshold":            thr,
            "alerts_sent":          alerts_sent,
            "true_at_risk_reached": true_at_risk,
            "false_alerts":         false_alerts,
            "advisor_hours":        advisor_hours,
        })

surface_df = pd.DataFrame(surface_rows)
surface_df.to_csv(
    OUT / "table_5_2_intervention_workload_simulation_by_week_threshold.csv",
    index=False)

pivot = surface_df.pivot(
    index="week",
    columns="threshold",
    values="true_at_risk_reached")

heatmap(pivot,
        title="Figure 5.2 Threshold surface for adaptive alerting",
        xlabel="Threshold",
        ylabel="Week",
        path=str(OUT / "figure_5_2_threshold_surface_for_adaptive_alerting.pdf"))

surface_df.head()

,week,threshold,alerts_sent,true_at_risk_reached,false_alerts,advisor_hours
0,2,0.3,1898,914,984,322.66
1,2,0.4,1011,592,419,171.87
2,2,0.5,614,417,197,104.38
3,2,0.6,358,280,78,60.86
4,2,0.7,263,219,44,44.71
